# Add xAAEnet Blocks to a User Encoder

> Add the xAAEnet blocks needed to explain a user-provided model.

In [ ]:
#| default_exp user_encoder

The goal of this module is to start from the user's own encoder and add the xAAEnet blocks defined in `01_model_aae.ipynb`. We do not replace the user's model idea; we add the components needed to analyze and explain it:

1. keep the user encoder as the feature extractor;
2. project encoder features into the latent space `z`;
3. add a label head from `z`;
4. add the latent discriminator that regularizes `z`;
5. build a symmetric U-Net decoder with `DynamicUnetSkipDropout` from the user encoder and its skip connections.

The encoder must therefore expose spatial feature maps that can be used by the U-Net hooks. For transformer models, the user should provide a wrapper that exposes a spatial feature-map encoder before adding these xAAEnet blocks.

In [ ]:
#| export
from collections.abc import Callable

import torch
import torch.nn as nn
import torch.nn.functional as F
from fastai.torch_core import TensorBase
from torch import Tensor

from tell_me_why.model_aae import DynamicUnetSkipDropout


def set_module_trainable(module: nn.Module, trainable: bool) -> nn.Module:
    """Enable or disable gradient updates for every parameter in a module."""
    for parameter in module.parameters():
        parameter.requires_grad = trainable
    return module


def _module_device(module: nn.Module) -> torch.device:
    try:
        return next(module.parameters()).device
    except StopIteration:
        return torch.device("cpu")


class EncoderWithAAEBlocks(nn.Module):
    """User encoder extended with xAAEnet analysis blocks.

    The user encoder stays the feature extractor. The added blocks project its
    features into `z`, predict from `z`, regularize `z`, and reconstruct the
    input with a symmetric U-Net decoder.
    """

    def __init__(
        self,
        encoder: nn.Module,
        input_size: int = 256,
        input_channels: int = 3,
        encoding_dims: int = 128,
        classes: int = 2,
        linear: nn.Module | None = None,
        gen_train: bool = True,
        skip_dropout: float = 1.0,
        freeze_encoder: bool = False,
    ):
        super().__init__()
        self.gen_train = gen_train
        self.input_size = input_size
        self.input_channels = input_channels
        self.encoding_dims = encoding_dims
        self.classes = classes

        if freeze_encoder:
            set_module_trainable(encoder, False)

        self.unet = DynamicUnetSkipDropout(
            encoder=encoder,
            n_out=input_channels,
            img_size=(input_size, input_size),
            skip_dropout=skip_dropout,
            last_cross=False,
        )

        self.flatten = nn.Flatten()
        self.encoder_feature_shape = self._infer_encoder_feature_shape()
        flat_features = int(torch.tensor(self.encoder_feature_shape).prod().item())

        self.fc_encode = nn.Linear(flat_features, encoding_dims)
        self.bn_lin = nn.BatchNorm1d(num_features=encoding_dims)
        self.decoder_fc = nn.Linear(encoding_dims, flat_features)
        self.linear = linear if linear is not None else nn.Linear(encoding_dims, self.classes)

        self.fc_crit1 = nn.Linear(encoding_dims, 64)
        self.fc_crit2 = nn.Linear(64, 16)
        self.fc_crit3 = nn.Linear(16, 1)
        self.bn_crit1 = nn.BatchNorm1d(num_features=64)
        self.bn_crit2 = nn.BatchNorm1d(num_features=16)

    def _infer_encoder_feature_shape(self) -> tuple[int, ...]:
        encoder = self.unet.layers[0]
        was_training = encoder.training
        device = _module_device(encoder)
        try:
            encoder.eval()
            with torch.no_grad():
                sample = torch.zeros(1, self.input_channels, self.input_size, self.input_size, device=device)
                features = encoder(sample)
        finally:
            encoder.train(was_training)
        return tuple(features.shape[1:])

    def latent_gan(self, z: Tensor) -> Tensor:
        x = F.leaky_relu(self.bn_crit1(self.fc_crit1(z)), negative_slope=0.2)
        x = F.leaky_relu(self.bn_crit2(self.fc_crit2(x)), negative_slope=0.2)
        return torch.sigmoid(self.fc_crit3(x))

    def reconstruction_loss(self, loss_func: Callable[[Tensor, Tensor], Tensor]) -> Tensor:
        return loss_func(self.decoder_output, self.input_image)

    def classif_loss_func(
        self,
        output: Tensor,
        target: Tensor,
        RECONS_WEIGHT: float = 0.0,
        CLASS_WEIGHT: float = 1.0,
        user_loss_func: Callable[..., Tensor] | None = None,
        reconstruction_loss_func: Callable[[Tensor, Tensor], Tensor] | None = None,
        **kwargs,
    ) -> Tensor:
        user_classif_loss = user_loss_func if user_loss_func is not None else F.cross_entropy
        self.classif_loss = user_classif_loss(output, target, **kwargs)
        self.recons_loss = (
            output.new_zeros(())
            if reconstruction_loss_func is None
            else self.reconstruction_loss(reconstruction_loss_func)
        )
        return CLASS_WEIGHT * self.classif_loss + RECONS_WEIGHT * self.recons_loss

    def aae_loss_func(
        self,
        output: Tensor,
        target: Tensor,
        RECONS_WEIGHT: float = 0.0,
        CLASS_WEIGHT: float = 1.0,
        ADV_WEIGHT: float = 1.0,
        user_loss_func: Callable[..., Tensor] | None = None,
        reconstruction_loss_func: Callable[[Tensor, Tensor], Tensor] | None = None,
        **kwargs,
    ) -> Tensor:
        adversarial_loss = nn.BCELoss()

        if self.gen_train:
            valid = torch.ones_like(self.gan_fake, requires_grad=False).detach()
            self.adv_loss = adversarial_loss(self.gan_fake, valid)
            self.crit_loss = output.new_zeros(())
        else:
            valid = torch.ones_like(self.gan_real, requires_grad=False).detach()
            fake = torch.zeros_like(self.gan_fake, requires_grad=False).detach()
            self.real_loss = adversarial_loss(self.gan_real, valid)
            self.fake_loss = adversarial_loss(self.gan_fake, fake)
            self.adv_loss = 0.6 * self.real_loss + 0.4 * self.fake_loss
            self.crit_loss = self.adv_loss

        user_classif_loss = user_loss_func if user_loss_func is not None else F.cross_entropy
        self.classif_loss = user_classif_loss(output, target, **kwargs)
        self.recons_loss = (
            output.new_zeros(())
            if reconstruction_loss_func is None
            else self.reconstruction_loss(reconstruction_loss_func)
        )

        return (
            ADV_WEIGHT * self.adv_loss
            + CLASS_WEIGHT * self.classif_loss
            + RECONS_WEIGHT * self.recons_loss
        )

    def forward(self, x: Tensor) -> Tensor:
        self.input_image = x

        feats = self.unet.layers[0](x)
        flat = self.flatten(feats)
        self.z = F.leaky_relu(self.bn_lin(self.fc_encode(flat)), negative_slope=0.2)

        labels = self.linear(self.z)
        self.gan_fake = self.latent_gan(self.z)
        self.gan_real = self.latent_gan(torch.randn_like(self.z))

        z_spatial = F.relu(self.decoder_fc(self.z))
        z_spatial = z_spatial.view(-1, *self.encoder_feature_shape)
        out = TensorBase(z_spatial)
        orig_x = TensorBase(torch.zeros_like(self.input_image))

        for layer in self.unet.layers[1:]:
            out.orig = orig_x
            nres = layer(out)
            out.orig = None
            if hasattr(nres, "orig"):
                nres.orig = None
            out = nres

        self.decoder_output = out
        return labels


def add_xaae_blocks(
    encoder: nn.Module,
    input_size: int = 256,
    input_channels: int = 3,
    encoding_dims: int = 128,
    classes: int = 2,
    linear: nn.Module | None = None,
    gen_train: bool = True,
    skip_dropout: float = 1.0,
    freeze_encoder: bool = False,
) -> EncoderWithAAEBlocks:
    """Add xAAEnet analysis blocks to a user encoder."""
    return EncoderWithAAEBlocks(
        encoder=encoder,
        input_size=input_size,
        input_channels=input_channels,
        encoding_dims=encoding_dims,
        classes=classes,
        linear=linear,
        gen_train=gen_train,
        skip_dropout=skip_dropout,
        freeze_encoder=freeze_encoder,
    )

## Practical Use Case

The user provides the encoder. `add_xaae_blocks` adds the xAAEnet analysis blocks and automatically builds the matching U-Net decoder through `DynamicUnetSkipDropout`.

The forward pass returns `labels`, while the model keeps `z`, `gan_fake`, `gan_real`, and `decoder_output` as attributes for training and analysis.

In [ ]:
user_encoder = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
)

model = add_xaae_blocks(
    encoder=user_encoder,
    input_size=32,
    input_channels=3,
    encoding_dims=16,
    classes=2,
)

batch = torch.randn(2, 3, 32, 32)
labels = model(batch)
labels.shape, model.z.shape, model.decoder_output.shape

## Training Loss

The supervised label loss keeps the same name as in `AAE`: `classif_loss_func`. This project focuses on binary classifiers, so the model returns two logits per sample and the default loss is cross-entropy.

The reconstruction loss is separate and must be passed explicitly when needed. For example, a project can pass the same `MS-SSIM + L1` objective used in `AAE`, but this notebook does not import or impose that loss. `aae_loss_func` combines the binary classification loss, the optional reconstruction loss, and the adversarial latent loss.

In [ ]:
target_labels = torch.tensor([0, 1])

classif_loss = model.classif_loss_func(labels, target_labels)

aae_loss = model.aae_loss_func(
    labels,
    target_labels,
    CLASS_WEIGHT=1.0,
    RECONS_WEIGHT=0.1,
    ADV_WEIGHT=0.1,
)

classif_loss.shape, aae_loss.shape

## Encoder Contract

The added blocks stay intentionally close to `AAE`: they expect an encoder that can be passed to `DynamicUnetSkipDropout`. In practice, that means a spatial encoder whose intermediate feature maps can be hooked for U-Net skip connections.

If a project uses a transformer, the user should wrap it so the xAAEnet blocks receive spatial feature maps. The notebook keeps that adapter outside this module because it depends on the transformer's architecture.